# Fetch results 

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
            library(SeuratWrappers), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # miloR
            library(miloR), 
            library(ggbeeswarm), 

            # Plotting 
            library(ggplot2), 
            library(patchwork), 
            library(cowplot), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv="pyscenic.0.12.1", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Import data 

In [ ]:
# Seurat object 
so <- readRDS("data/object/pp.rds")

In [ ]:
# AUC mtx from individual grn 
auc_mtx_0 <- read.csv("result/scenic/res_0/auc_mtx.csv", row.names=1) %>% t()
rownames(auc_mtx_0) <- gsub("\\.", "", rownames(auc_mtx_0))

In [ ]:
# Attach AUC to Seurat 
so_0 <- subset(so, subset=cell_id %in% colnames(auc_mtx_0))
auc_mtx_0 <- auc_mtx_0[, colnames(so_0)]
so_0[["AUC"]] <- CreateAssayObject(data=auc_mtx_0[, colnames(so_0)], min.cells=0, min.features=0)

# TRM subsets annotation 

In [ ]:
t_cell_subset = read.csv("result/annotation/t_cell_subset.csv", row.names=1)
so_0 <- AddMetaData(so_0, t_cell_subset)

# RSS 

In [ ]:
meta <- so_0@meta.data

In [ ]:
# Load python libraries
pysenic_rss <- import("pyscenic.rss", convert=FALSE)
pd <- import("pandas", convert=FALSE)

In [ ]:
rss_get <- function(meta, auc_mtx, time_point=NULL) {
    
    # Load python libraries
    pysenic_rss <- import("pyscenic.rss", convert=FALSE)
    pd <- import("pandas", convert=FALSE)

    # Get meta subset
    if(!is.null(time_point)) {meta <- meta[meta$time_point==time_point, ]}
    if(is.null(time_point)) {time_point <- "ALL"}

    # Get AUC matx 
    auc_mtx <- auc_mtx[, rownames(meta)] %>% t() %>% as.data.frame()

    # Get regulon specific scores 
    rss <- pysenic_rss$regulon_specificity_scores(auc_mtx, pd$Series(r_to_py(meta$celltype_low))) %>% py_to_r()

    # Get matrix
    mat <- tibble::rownames_to_column(rss, "celltype_low") %>% 
        reshape2::melt(., id.vars=c("celltype_low"), variable.name="module", value.name="rss") %>% 
        dplyr::arrange(celltype_low, desc(rss)) %>% 
        dplyr::group_by(celltype_low) %>%
        dplyr::mutate(rank=1:n()) %>% 
        dplyr::mutate(label_rank=ifelse(rank %in% 1:15, as.character(module), NA)) %>% 
        dplyr::mutate(time_point=time_point)

    return(mat)
    
}

In [ ]:
rss_plot <- function(mat, title=NULL) {

    mat$celltype_low <- factor(mat$celltype_low, levels=names(color$celltype_low_r))

    dp_rss_top_label <- ggplot(mat, aes(x=rank, y=rss, label=label_rank, color=celltype_low)) + 
        geom_point(size=1) + 
        xlab("") + ylab("RSS") + ggtitle(title) + 
        ylim(0, 1) + 
        facet_grid(~celltype_low, scales="free_y") +
        scale_color_manual(values=color$celltype_low_r, name="Cell type") + 
        scale_x_continuous(breaks=seq(0, floor(max(mat_1$rank)/100)*100, 100), labels=seq(0, floor(max(mat$rank)/100)*100, 100)) + 
        ggrepel::geom_text_repel(segment.color="black", force=10, force_pull=1, max.overlaps=getOption("ggrepel.max.overlaps", default=100), size=5, alpha=1, guide="none", segment.size=0.05, color="black", min.segment.length=0) + 
        guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
        facet_wrap(~celltype_low, nrow=1, labeller = label_parsed) + 
        theme(legend.position="none")

    return(dp_rss_top_label)
    
}

In [ ]:
mat_0 <- rss_get(so_0@meta.data, auc_mtx_0, "Baseline")
mat_1 <- rss_get(so_0@meta.data, auc_mtx_0, "Tx")
mat_2 <- rss_get(so_0@meta.data, auc_mtx_0, "D14")
mat_3 <- rss_get(so_0@meta.data, auc_mtx_0, "D100")
mat_4 <- rss_get(so_0@meta.data, auc_mtx_0, "GVHD")
mat_5 <- rss_get(so_0@meta.data, auc_mtx_0)
mat_6 <- rss_get(so_0@meta.data %>% dplyr::mutate(celltype_low=t_cell_subset) %>% dplyr::filter(!is.na(celltype_low)), auc_mtx_0)

In [ ]:
saveRDS(
    
    list(
    
        Baseline=mat_0, 
        Tx=mat_1, 
        D14=mat_2, 
        D100=mat_3, 
        GVHD=mat_4,
        ALL=mat_5
        
    ), 
    
    "result/scenic/res_0/rss.rds"

)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)

rss_plot(mat_5, "T cell (Over all)") 
rss_plot(mat_0, "T cell (Baseline)")
rss_plot(mat_1, "T cell (Tx)") 
rss_plot(mat_2, "T cell (D14)") 
rss_plot(mat_3, "T cell (D100)") 
rss_plot(mat_4, "T cell (GVHD)") 

# AUC Wilcoxon  

In [ ]:
wilcox <- function(so, ident, ident_1=NULL, ident_2=NULL, only_pos=FALSE, avg_log2FC_threshold=0, min_pct=0, test_use="wilcox", assay="RNA", slot="data") {
    
    # Drop empty levels 
    so@meta.data <- droplevels(so@meta.data)
    
    # Check number of cells 
    n_cells_1 <- sum(so@meta.data[[ident]]==ident_1)
    n_cells_2 <- sum(so@meta.data[[ident]]==ident_2)

    check_1 <- n_cells_1 >= 3
    check_2 <- n_cells_2 >= 3
    
    if(check_1 & check_2) {
        
        so <- SetIdent(so, value=ident)
        res <- RunPresto(so, ident.1=ident_1, ident.2=ident_2, logfc.threshold=avg_log2FC_threshold, min.pct=min_pct, only.pos=only_pos, test.use=test_use, assay=assay, slot=slot)
        
        # Annotate results 
        res$gene <- rownames(res)

        # N cells per group 
        res$n_cells_1 <- n_cells_1
        res$n_cells_2 <- n_cells_2

        # Cell type 
        res$celltype_low <- so$celltype_low[1]

        return(res)
        
    } else {
        
        return(NULL)
        
    }

}

# Time points 

In [ ]:
so_1 <- SplitObject(so_0, split.by="celltype_low")

## Baseline vs Tx

In [ ]:
contrasts_vec <- c("Tx", "Baseline")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=time_point %in% contrasts_vec))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="time_point", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_1[mat_1$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_0[mat_0$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- lapply(res_1, function(x) x$celltype_low[1]) %>% unlist()

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)

## Baseline vs D14

In [ ]:
contrasts_vec <- c("D14", "Baseline")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=time_point %in% contrasts_vec))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="time_point", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_2[mat_2$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_0[mat_0$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- lapply(res_1, function(x) x$celltype_low[1]) %>% unlist()

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)

## Baseline vs D100

In [ ]:
contrasts_vec <- c("D100", "Baseline")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=time_point %in% contrasts_vec))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="time_point", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_3[mat_3$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_0[mat_0$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- lapply(res_1, function(x) x$celltype_low[1]) %>% unlist()

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)

## Baseline vs GVHD

In [ ]:
contrasts_vec <- c("GVHD", "Baseline")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=time_point %in% contrasts_vec))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="time_point", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_4[mat_4$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_0[mat_0$celltype_low==i, c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- lapply(res_1, function(x) x$celltype_low[1]) %>% unlist()

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)

## Th17hi vs Th17lo 

In [ ]:
so_1 <- SplitObject(so_0, split.by="hto_demux_class")

In [ ]:
contrasts_vec <- c("Th17hi", "Th17lo")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=t_cell_subset %in% contrasts_vec & time_point=="Baseline"))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="t_cell_subset", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
hto_demux_class <- names(res_1)

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_6[mat_6$celltype_low==contrasts_vec[1], c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_6[mat_6$celltype_low==contrasts_vec[2], c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- hto_demux_class

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)

## Tc17hi vs Tc17lo 

In [ ]:
so_1 <- SplitObject(so_0, split.by="hto_demux_class")

In [ ]:
contrasts_vec <- c("Tc17hi", "Tc17lo")

In [ ]:
so_tmp <- so_1
so_tmp <- lapply(so_tmp, function(x) subset(x, subset=t_cell_subset %in% contrasts_vec & time_point=="Baseline"))
so_tmp <- so_tmp[!lapply(so_tmp, is.null) %>% unlist()]

In [ ]:
res_1 <- lapply(so_tmp, function(so) {wilcox(
    
    so=so, 
    ident="t_cell_subset", 
    ident_1=contrasts_vec[1], 
    ident_2=contrasts_vec[2], 
    only_pos=FALSE, 
    avg_log2FC_threshold=0, 
    min_pct=0, 
    assay="AUC", 
    slot="data", 
    test_use="wilcox"
    
)
                                  }
               )

In [ ]:
hto_demux_class <- names(res_1)

In [ ]:
res_1 <- lapply(names(res_1), function(i) {

    res <- res_1[[i]] %>% dplyr::rename(module=gene)

    rss <- dplyr::inner_join(
        
        mat_6[mat_6$celltype_low==contrasts_vec[1], c("module", "rss", "rank")] %>% dplyr::rename(rss_1=rss, rank_1=rank), 
        mat_6[mat_6$celltype_low==contrasts_vec[2], c("module", "rss", "rank")] %>% dplyr::rename(rss_2=rss, rank_2=rank), 
        by=join_by(module)
    
    )

    res <- dplyr::left_join(res, rss, by=join_by(module))

    rownames(res) <- res$module

    return(res)
    
}
              )

In [ ]:
names(res_1) <- hto_demux_class

In [ ]:
names(res_1) <- make.names(names(res_1))
saveRDS(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".rds"))
openxlsx::write.xlsx(res_1, paste0("result/scenic/wilcox/", "auc_", tolower(contrasts_vec[1]), "_vs_", tolower(contrasts_vec[2]), ".xlsx"), colNames=TRUE, rowNames=TRUE)